In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 🧠 Відкриваємо з'єднання з DuckDB
con = duckdb.connect()

# 📁 Шлях до файлу
parquet_path = r"/mnt/c/Users/5302/PycharmProjects/geoid/data/NMAD_dem.parquet"

# 🎯 LULC класи для вибірки
lulc_classes = (1, 2, 5, 7, 11)

# 🧾 SQL-запит: фільтрація та вибір тільки потрібних колонок
query = f"""
SELECT
    lulc_class,
    nmad_alos, nmad_aster, nmad_cop,
    nmad_fab, nmad_nasa, nmad_srtm, nmad_tan
FROM '{parquet_path}'
WHERE lulc_class IN {lulc_classes}
"""

grouped = con.execute(query).fetchdf()
con.close()

# 🏷️ Мапінг назв
lulc_map = {
    1: "Water",
    2: "Trees",
    5: "Crops",
    7: "Built Area",
    11: "Rangeland"
}
grouped["LULC"] = grouped["lulc_class"].map(lulc_map)

# 🔁 Перетворення в long-form
nmad_cols = ["nmad_alos", "nmad_aster", "nmad_cop",
             "nmad_fab", "nmad_nasa", "nmad_srtm", "nmad_tan"]

df_long = grouped.melt(
    id_vars="LULC",
    value_vars=nmad_cols,
    var_name="DEM",
    value_name="NMAD"
)
df_long["DEM"] = df_long["DEM"].str.replace("nmad_", "").str.upper()

# === ВІЗУАЛІЗАЦІЯ ===
sns.set(style="whitegrid", context="talk", font="serif")

plt.figure(figsize=(10, 6))
sns.barplot(
    data=df_long,
    x="LULC", y="NMAD", hue="DEM",
    palette="tab10", edgecolor="black"
)

plt.xlabel("Land Use / Land Cover Class", fontsize=13)
plt.ylabel("Normalized Median Absolute Deviation (m)", fontsize=13)
plt.legend(title="Digital Elevation Model (DEM)", bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=10)
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()



In [ ]:
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- 1. Load all unique LULC classes and compute NMAD means ---
path = r"/mnt/c/Users/5302/PycharmProjects/geoid/data/NMAD_dem.parquet"
con = duckdb.connect()

# Автоматичне визначення класів
unique_classes = con.execute(f"""
    SELECT DISTINCT lulc_class
    FROM '{path}'
    WHERE lulc_class IS NOT NULL
""").fetchdf()
lulc_values = tuple(unique_classes["lulc_class"].sort_values())

# Обчислення середніх NMAD для всіх DEM по класах
query = f"""
SELECT
    lulc_class,
    AVG(nmad_alos) AS "ALOS",
    AVG(nmad_aster) AS "ASTER",
    AVG(nmad_cop) AS "Copernicus",
    AVG(nmad_fab) AS "FABDEM",
    AVG(nmad_nasa) AS "NASADEM",
    AVG(nmad_srtm) AS "SRTM",
    AVG(nmad_tan) AS "TanDEM-X"
FROM '{path}'
WHERE lulc_class IN {lulc_values}
GROUP BY lulc_class
ORDER BY lulc_class
"""
df = con.execute(query).fetchdf()
con.close()

# --- 2. Map class labels (гарантовано виведе назву навіть якщо немає у словнику) ---
lulc_map = {
    0: "No Data",
    1: "Water",
    2: "Trees",
    4: "Flooded Vegetation",
    5: "Crops",
    7: "Built Area",
    8: "Bare Ground",
    9: "Snow/Ice",
    10: "Clouds",
    11: "Rangeland"
}
df["LULC"] = df["lulc_class"].map(lulc_map).fillna(df["lulc_class"].astype(str))
df = df.set_index("LULC").drop(columns="lulc_class")

# --- 3. Optional: reorder by average NMAD (to highlight best classes first) ---
df = df.loc[df.mean(axis=1).sort_values().index]

# --- 4. Heatmap ---
sns.set_theme(style="white", context="paper", font="serif")

plt.figure(figsize=(9, 5))
ax = sns.heatmap(
    df,
    annot=True, fmt=".2f",
    cmap="viridis",  # _r = reverse (dark low values, light high)
    linewidths=0.5,
    cbar_kws={"label": "Normalized Median Absolute Deviation (NMAD, m)"}
)

# --- 5. Aesthetic tuning for publication ---
ax.set_xlabel("Digital Elevation Model (DEM)", fontsize=12)
ax.set_ylabel("Land Use / Land Cover Class", fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.tick_params(axis='both', labelsize=10)
plt.tight_layout()

plt.show()


In [ ]:
def compute_grouped_table(path, group_col):
    con = duckdb.connect()
    nmad_cols = [
        "nmad_alos", "nmad_aster", "nmad_cop",
        "nmad_fab", "nmad_nasa", "nmad_srtm", "nmad_tan"
    ]
    cols_str = ",\n".join([
        f"AVG({c}) AS \"{c.replace('nmad_', '').upper()}\"" for c in nmad_cols
    ])

    dtype = con.execute(
        f"SELECT typeof({group_col}) FROM '{path}' WHERE {group_col} IS NOT NULL LIMIT 1"
    ).fetchone()[0]

    # --- категоріальні (LULC, Landform)
    if dtype in ("INTEGER", "BIGINT"):
        query = f"""
        SELECT
            {group_col} AS class,
            {cols_str}
        FROM '{path}'
        WHERE {group_col} IS NOT NULL
        GROUP BY {group_col}
        ORDER BY {group_col}
        """

    # --- Slope (INSPIRE)
    elif "slope" in group_col.lower():
        query = f"""
        SELECT
            CASE
                WHEN {group_col} < 2 THEN 'Flat (0–2°)'
                WHEN {group_col} < 6 THEN 'Undulating (2–6°)'
                WHEN {group_col} < 12 THEN 'Hilly (6–12°)'
                ELSE 'Mountainous (>12°)'
            END AS class,
            {cols_str}
        FROM '{path}'
        WHERE {group_col} IS NOT NULL
        GROUP BY class
        ORDER BY MIN({group_col})
        """

    # --- HAND
    elif "hand" in group_col.lower() or "atl08" in group_col.lower():
        query = f"""
        SELECT
            CASE
                WHEN {group_col} < 2 THEN '0–2 m'
                WHEN {group_col} < 5 THEN '2–5 m'
                WHEN {group_col} < 10 THEN '5–10 m'
                WHEN {group_col} < 20 THEN '10–20 m'
                ELSE '>20 m'
            END AS class,
            {cols_str}
        FROM '{path}'
        WHERE {group_col} IS NOT NULL
        GROUP BY class
        ORDER BY MIN({group_col})
        """

    # --- TWI (Drover et al. 2015)
    elif "twi" in group_col.lower():
        query = f"""
        SELECT
            CASE
                WHEN {group_col} < 10 THEN 'Very Dry (<10%)'
                WHEN {group_col} < 25 THEN 'Dry (10–25%)'
                WHEN {group_col} < 75 THEN 'Medium (25–75%)'
                WHEN {group_col} < 90 THEN 'Wet (75–90%)'
                ELSE 'Very Wet (>90%)'
            END AS class,
            {cols_str}
        FROM '{path}'
        WHERE {group_col} IS NOT NULL
        GROUP BY class
        ORDER BY MIN({group_col})
        """

    else:
        query = f"""
        SELECT
            {group_col} AS class,
            {cols_str}
        FROM '{path}'
        WHERE {group_col} IS NOT NULL
        GROUP BY {group_col}
        ORDER BY {group_col}
        """

    df = con.execute(query).fetchdf()
    con.close()
    return df


In [ ]:
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# === 1. Шлях до parquet ===
path = r"/mnt/c/Users/5302/PycharmProjects/geoid/data/NMAD_dem.parquet"

# === 3. Виклики ===

df_lulc  = compute_grouped_table(path, "lulc_class")
df_slope = compute_grouped_table(path, "copernicus_dem_slope")
df_geom  = compute_grouped_table(path, "copernicus_dem_landform")
df_hand  = compute_grouped_table(path, "hand_m")
df_twi   = compute_grouped_table(path, "copernicus_dem_twi")





In [ ]:
def plot_heatmap(df, title):
    sns.set_theme(style="white", context="paper", font="serif")
    plt.figure(figsize=(9, 5))
    df_p = df.set_index("Class").drop(columns="class")
    sns.heatmap(df_p, annot=True, fmt=".2f", cmap="viridis",
                linewidths=0.5, cbar_kws={"label": "NMAD (m)"})
    plt.title(title, fontsize=13)
    plt.tight_layout()
    plt.show()

plot_heatmap(df_lulc, "NMAD by Land Use / Land Cover")
plot_heatmap(df_slope, "NMAD by Slope (INSPIRE classes)")
plot_heatmap(df_geom, "NMAD by Landform (Geomorphon)")
plot_heatmap(df_hand, "NMAD by HAND (Height Above Drainage)")
plot_heatmap(df_twi, "NMAD by Topographic Wetness Index (Drover quantiles)")


In [ ]:
import duckdb

path = r"/mnt/c/Users/5302/PycharmProjects/geoid/data/NMAD_dem.parquet"
con = duckdb.connect()
cols = con.execute(f"DESCRIBE SELECT * FROM '{path}'").fetchdf()
print(cols["column_name"].tolist())
con.close()

In [ ]:
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

path = r"/mnt/c/Users/5302/PycharmProjects/geoid/data/NMAD_dem.parquet"
con = duckdb.connect()

# === 1️⃣ DEM-словник
dems = {
    "alos": "ALOS",
    "aster": "ASTER",
    "cop": "Copernicus",   # коротка форма для NMAD
    "fab": "FABDEM",
    "nasa": "NASADEM",
    "srtm": "SRTM",
    "tan": "TanDEM-X"
}

# ---------------------------------------------------------------------
# 2️⃣ LULC
cols_str = ", ".join([f"AVG(nmad_{key}) AS \"{val}\"" for key, val in dems.items()])
sql_lulc = (
    "SELECT lulc_class AS label, "
    f"{cols_str} "
    f"FROM '{path}' "
    "WHERE lulc_class IS NOT NULL "
    "GROUP BY lulc_class "
    "ORDER BY lulc_class"
)
df_lulc = con.execute(sql_lulc).fetchdf()

lulc_map = {
    0: "No Data", 1: "Water", 2: "Trees", 4: "Flooded Vegetation",
    5: "Crops", 7: "Built Area", 8: "Bare Ground", 9: "Snow/Ice",
    10: "Clouds", 11: "Rangeland"
}
df_lulc["label"] = df_lulc["label"].map(lulc_map).fillna(df_lulc["label"].astype(str))

# ---------------------------------------------------------------------
# 3️⃣ Slope
parts = []
for key, val in dems.items():
    prefix = "copernicus" if key == "cop" else key
    slope_col = f"{prefix}_dem_slope"
    sql = (
        "SELECT "
        f"'{val}' AS DEM, "
        "CASE "
        f"WHEN {slope_col} < 2 THEN 'Flat (0–2°)' "
        f"WHEN {slope_col} < 6 THEN 'Undulating (2–6°)' "
        f"WHEN {slope_col} < 12 THEN 'Hilly (6–12°)' "
        "ELSE 'Mountainous (>12°)' END AS label, "
        f"AVG(nmad_{key}) AS nmad "
        f"FROM '{path}' "
        f"WHERE {slope_col} IS NOT NULL "
        "GROUP BY label"
    )
    parts.append(con.execute(sql).fetchdf())
df_slope = pd.concat(parts)
df_slope = df_slope.pivot(index="label", columns="DEM", values="nmad").reset_index()
order_slope = [
    "Flat (0–2°)",
    "Undulating (2–6°)",
    "Hilly (6–12°)",
    "Mountainous (>12°)"
]
df_slope["label"] = pd.Categorical(df_slope["label"], categories=order_slope, ordered=True)
df_slope = df_slope.sort_values("label")

# ---------------------------------------------------------------------
# 4️⃣ Landform
geom_names = {1:"Flat",2:"Peak",3:"Ridge",4:"Shoulder",5:"Spur",
              6:"Slope",7:"Hollow",8:"Footslope",9:"Valley",10:"Pit"}
parts = []
for key, val in dems.items():
    prefix = "copernicus" if key == "cop" else key
    landform_col = f"{prefix}_dem_landform"
    sql = (
        "SELECT "
        f"'{val}' AS DEM, "
        f"{landform_col} AS code, "
        f"AVG(nmad_{key}) AS nmad "
        f"FROM '{path}' "
        f"WHERE {landform_col} IS NOT NULL "
        f"GROUP BY {landform_col}"
    )
    df = con.execute(sql).fetchdf()
    df["label"] = df["code"].map(geom_names).fillna(df["code"].astype(str))
    parts.append(df)
df_geom = pd.concat(parts)
df_geom = df_geom.pivot(index="label", columns="DEM", values="nmad").reset_index()

# ---------------------------------------------------------------------
# 5️⃣ TWI
parts = []
for key, val in dems.items():
    prefix = "copernicus" if key == "cop" else key
    twi_col = f"{prefix}_dem_twi"
    sql = (
        "SELECT "
        f"'{val}' AS DEM, "
        "CASE "
        f"WHEN {twi_col} < 5 THEN 'Very Low' "
        f"WHEN {twi_col} < 7 THEN 'Low' "
        f"WHEN {twi_col} < 9 THEN 'Medium' "
        f"WHEN {twi_col} < 11 THEN 'High' "
        "ELSE 'Very High' END AS label, "
        f"AVG(nmad_{key}) AS nmad "
        f"FROM '{path}' "
        f"WHERE {twi_col} IS NOT NULL "
        "GROUP BY label"
    )
    parts.append(con.execute(sql).fetchdf())
df_twi = pd.concat(parts)
df_twi = df_twi.pivot(index="label", columns="DEM", values="nmad").reset_index()

# ---------------------------------------------------------------------
# 6️⃣ HAND (Height Above Drainage)
parts = []
for key, val in dems.items():
    prefix = "copernicus" if key == "cop" else key
    hand_col = f"{prefix}_dem_2000"
    sql = (
        "SELECT "
        f"'{val}' AS DEM, "
        "CASE "
        f"WHEN {hand_col} < 2 THEN '0–2 m' "
        f"WHEN {hand_col} < 5 THEN '2–5 m' "
        f"WHEN {hand_col} < 10 THEN '5–10 m' "
        f"WHEN {hand_col} < 20 THEN '10–20 m' "
        "ELSE '>20 m' END AS label, "
        f"AVG(nmad_{key}) AS nmad "
        f"FROM '{path}' "
        f"WHERE {hand_col} IS NOT NULL "
        "GROUP BY label"
    )
    parts.append(con.execute(sql).fetchdf())
df_hand = pd.concat(parts)
df_hand = df_hand.pivot(index="label", columns="DEM", values="nmad").reset_index()

con.close()

# ---------------------------------------------------------------------
# 7️⃣ Heatmap plotting
def plot_heatmap(df, title):
    sns.set_theme(style="white", context="paper", font="serif")
    plt.figure(figsize=(9,5))
    df_p = df.set_index("label")
    sns.heatmap(df_p, annot=True, fmt=".2f", cmap="viridis",
                linewidths=0.5, cbar_kws={"label": "NMAD (m)"})
    plt.title(title, fontsize=12)
    plt.xlabel("Digital Elevation Model (DEM)")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------------------
plot_heatmap(df_lulc,  "(e) Land Use / Land Cover")
plot_heatmap(df_slope, "(d) Slope")
plot_heatmap(df_geom,  "(c) Landform")
plot_heatmap(df_twi,   "(a) TWI (Topographic Wetness Indices)")
plot_heatmap(df_hand,  "(b) HAND (Height Above Drainage)")


In [2]:
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ============================================================
# SETTINGS

path = r"/mnt/c/Users/5302/PycharmProjects/geoid/data/NMAD_dem.parquet"

sns.set_theme(style="white", context="paper", font="serif")

VMIN = 2
VMAX = 12

# ============================================================
# DEM dictionary

dems = {
    "alos": "ALOS",
    "aster": "ASTER",
    "cop": "Copernicus",
    "fab": "FABDEM",
    "nasa": "NASADEM",
    "srtm": "SRTM",
    "tan": "TanDEM-X"
}

con = duckdb.connect()

# ============================================================
# FORMAT N

def format_n(n):

    if n >= 1_000_000:
        return f"{n/1_000_000:.2f}M"

    if n >= 1000:
        return f"{n/1000:.0f}k"

    return str(n)

# ============================================================
# GENERIC SQL RUNNER

def compute_factor(sql_template):

    parts = []

    for key, val in dems.items():

        prefix = "copernicus" if key == "cop" else key

        sql = sql_template.format(
            DEM_NAME=val,
            PREFIX=prefix,
            KEY=key
        )

        parts.append(con.execute(sql).fetchdf())

    df_raw = pd.concat(parts)

    df_val = df_raw.pivot(
        index="label",
        columns="DEM",
        values="nmad"
    )

    df_n = df_raw.pivot(
        index="label",
        columns="DEM",
        values="n"
    )

    return df_val, df_n

# ============================================================
# SLOPE

parts = []

for key, val in dems.items():

    prefix = "copernicus" if key == "cop" else key
    slope_col = f"{prefix}_dem_slope"

    sql = (
        "SELECT "
        f"'{val}' AS DEM, "
        "CASE "
        f"WHEN {slope_col} < 2 THEN 'Flat (0–2°)' "
        f"WHEN {slope_col} < 6 THEN 'Undulating (2–6°)' "
        f"WHEN {slope_col} < 12 THEN 'Hilly (6–12°)' "
        "ELSE 'Mountainous (>12°)' END AS label, "
        f"AVG(nmad_{key}) AS nmad, "
        "COUNT(*) AS n "
        f"FROM '{path}' "
        f"WHERE {slope_col} IS NOT NULL "
        "GROUP BY label"
    )

    parts.append(con.execute(sql).fetchdf())

df_slope_raw = pd.concat(parts)

order_slope = [
    "Flat (0–2°)",
    "Undulating (2–6°)",
    "Hilly (6–12°)",
    "Mountainous (>12°)"
]

df_slope_val = df_slope_raw.pivot(index="label", columns="DEM", values="nmad")
df_slope_n = df_slope_raw.pivot(index="label", columns="DEM", values="n")

df_slope_val = df_slope_val.reindex(order_slope)
df_slope_n = df_slope_n.reindex(order_slope)

# ============================================================
# TWI

sql_twi = f"""
SELECT
    '{{DEM_NAME}}' AS DEM,
    CASE
        WHEN {{PREFIX}}_dem_twi < 5 THEN 'Very Low'
        WHEN {{PREFIX}}_dem_twi < 7 THEN 'Low'
        WHEN {{PREFIX}}_dem_twi < 9 THEN 'Medium'
        WHEN {{PREFIX}}_dem_twi < 11 THEN 'High'
        ELSE 'Very High'
    END AS label,
    AVG(nmad_{{KEY}}) AS nmad,
    COUNT(*) AS n
FROM '{path}'
WHERE {{PREFIX}}_dem_twi IS NOT NULL
GROUP BY label
"""

df_twi_val, df_twi_n = compute_factor(sql_twi)

# ============================================================
# HAND

parts = []

for key, val in dems.items():

    prefix = "copernicus" if key == "cop" else key
    hand_col = f"{prefix}_dem_2000"

    sql = (
        "SELECT "
        f"'{val}' AS DEM, "
        "CASE "
        f"WHEN {hand_col} < 2 THEN '0–2 m' "
        f"WHEN {hand_col} < 5 THEN '2–5 m' "
        f"WHEN {hand_col} < 10 THEN '5–10 m' "
        f"WHEN {hand_col} < 20 THEN '10–20 m' "
        "ELSE '>20 m' END AS label, "
        f"AVG(nmad_{key}) AS nmad, "
        "COUNT(*) AS n "
        f"FROM '{path}' "
        f"WHERE {hand_col} IS NOT NULL "
        "GROUP BY label"
    )

    parts.append(con.execute(sql).fetchdf())

df_hand_raw = pd.concat(parts)

order_hand = [
    "0–2 m",
    "2–5 m",
    "5–10 m",
    "10–20 m",
    ">20 m"
]

df_hand_val = df_hand_raw.pivot(index="label", columns="DEM", values="nmad")
df_hand_n = df_hand_raw.pivot(index="label", columns="DEM", values="n")

df_hand_val = df_hand_val.reindex(order_hand)
df_hand_n = df_hand_n.reindex(order_hand)
# ============================================================
# LANDFORM


sql_geom = f"""
SELECT
    '{{DEM_NAME}}' AS DEM,
    {{PREFIX}}_dem_landform AS label,
    AVG(nmad_{{KEY}}) AS nmad,
    COUNT(*) AS n
FROM '{path}'
WHERE {{PREFIX}}_dem_landform IS NOT NULL
GROUP BY label
"""

df_geom_val, df_geom_n = compute_factor(sql_geom)

# ============================================================
# LULC

sql_lulc = f"""
SELECT
    '{{DEM_NAME}}' AS DEM,
    CASE
        WHEN lulc_class = 1 THEN 'Water'
        WHEN lulc_class = 2 THEN 'Trees'
        WHEN lulc_class = 5 THEN 'Crops'
        WHEN lulc_class = 7 THEN 'Built Area'
        WHEN lulc_class = 11 THEN 'Rangeland'
        ELSE 'Other'
    END AS label,
    AVG(nmad_{{KEY}}) AS nmad,
    COUNT(*) AS n
FROM '{path}'
WHERE lulc_class IS NOT NULL
GROUP BY label
"""

df_lulc_val, df_lulc_n = compute_factor(sql_lulc)



con.close()




In [5]:
def plot_heatmap(values, counts, title, filename):

    plt.rcParams["svg.fonttype"] = "none"

    counts_fmt = counts.map(format_n)

    annot = values.round(2).astype(str) + "\n(n=" + counts_fmt + ")"

    rows, cols = values.shape

    cell_w = 1.2
    cell_h = 1.0

    plt.figure(figsize=(cols * cell_w + 2, rows * cell_h + 1))

    sns.heatmap(
        values,
        annot=annot,
        fmt="",
        cmap="viridis",
        vmin=VMIN,
        vmax=VMAX,
        linewidths=0.6,
        linecolor="white",
        square=True,
        annot_kws={"size":10},
        cbar_kws={"label": "NMAD (m)"}
    )

    plt.title(title, fontsize=12)
    plt.xlabel("Digital Elevation Model (DEM)")
    plt.ylabel("")

    plt.tight_layout()

    # journal formats
    plt.savefig(filename + ".tif", dpi=600, bbox_inches="tight")
    plt.savefig(filename + ".png", dpi=600, bbox_inches="tight")
    plt.savefig(filename + ".eps", bbox_inches="tight")
    plt.savefig(filename + ".svg", bbox_inches="tight")

    plt.close()

In [6]:
# ============================================================
# FIGURE PANELS

plot_heatmap(
    df_twi_val,
    df_twi_n,
    "(a) Topographic Wetness Index (TWI)",
    "figure5a_TWI"
)

plot_heatmap(
    df_hand_val,
    df_hand_n,
    "(b) HAND (Height Above Drainage)",
    "figure5b_HAND"
)

plot_heatmap(
    df_geom_val,
    df_geom_n,
    "(c) Landform",
    "figure5c_Landform"
)

plot_heatmap(
    df_slope_val,
    df_slope_n,
    "(d) Slope",
    "figure5d_Slope"
)

plot_heatmap(
    df_lulc_val,
    df_lulc_n,
    "(e) Land Use / Land Cover",
    "figure5e_LULC"
)